# Estimator order — A, B and C, SIL and bench

The analysis half of `README.md`: every capture under `sil/` and `bench/` is
loaded, the plan's metrics are tabulated per run, the correction residual is
decomposed into its physical terms, and each scenario is drawn with its
designs overlaid. SIL captures carry the plant's angle and velocity as truth;
bench captures are read through the residual and the encoder's own
finite-difference velocity, the two measures both sides share.

**Inputs:** `sil/<scenario>_<design>_J<factor>[_Tc<factor>].csv` from
`run_scenarios.sh` (one row per PWM cycle, regenerable; `sil/provenance_*.txt`
names the commit) and `bench/<scenario>_<design>[_Jcfg<J>][_r<n>].csv` exported
from the app, `Jcfg` the configured inertia in 1e-4 kg·m² when it is not the
board's default (the fast tier every cycle, the slower tiers leaving empty cells that are
carried forward here). Columns are DWARF paths, the same in both; a design-C
capture carries the torque state T̂_d as a third `x_hat` column (the A, B
and Bc captures from the rewritten head carry it too, at zero). A capture
holding several runs of one scenario, separated by at least half a second at
zero target, is split into them (`r1.1`, `r1.2`, …); every run's time is
rebased so its first nonzero target sits at 0.1 s.

In [ ]:
import csv
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path(".").resolve()
EST = "app_rotorEstimation_data.channels[0]"
MC = "app_motorControl_data.channels[0]"
COLS = {
    "theta_hat": f"{EST}.x_hat.e[0][0]",
    "omega_hat": f"{EST}.x_hat.e[1][0]",
    "tau_hat": f"{EST}.x_hat.e[2][0]",
    "theta_m": f"{EST}.angleMeasured_rad",
    "e": f"{EST}.angleError_rad",
    "u": f"{EST}.u",
    "i_q": f"{MC}.iQ_a",
    "tau_set": f"{MC}.torqueSetpoint_Nm",
    "omega_set": f"{MC}.velocitySetpoint_radPerSec",
    "theta_plant": "plant.angle_rad",
    "omega_plant": "plant.velocity_radps",
}
# The estimator and FOC configuration the captures ran with (app_rotorEstimation_channels.c,
# app_motorControl_channels.c as of 2026-11-07); a capture set at another config
# reads with its own values here.
OMEGA_N = 2 * np.pi * 50.0
ZETA = 1.0
P_D = 157.0
K_I = OMEGA_N**2
K_T = 0.825
J_CFG = 1.6e-4
B_VISC = 7.2e-4
T_C = 10.0e-3   # the configured Coulomb friction; the SIL plant's own is 4.4 mNm
# <scenario>_<design>[_J<plant factor>][_Tc<factor>][_Jcfg<configured J, 1e-4 kg m^2>][_r<n>]
# Designs: A, B, Bc (B with the Coulomb term: C with its torque state held at zero), C.
RUN_NAME = re.compile(r"^(S\d_\w+?)_(A|Bc?|C)(?:_J([\d.]+))?(?:_Tc([\d.]+))?(?:_Jcfg([\d.]+))?(?:_r(\d+))?\.csv$")
LEAD_S = 0.1   # where the first nonzero target sits after rebasing


def ffill(x):
    # Carry each sample forward over the empty cells a slower tier leaves.
    mask = ~np.isnan(x)
    if not mask.any():
        return x
    idx = np.maximum.accumulate(np.where(mask, np.arange(len(x)), 0))
    out = x[idx]
    out[: np.argmax(mask)] = np.nan
    return out


def load(path):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [[float(c) if c else np.nan for c in row] for row in reader]
    a = np.array(rows)
    idx = {h: k for k, h in enumerate(header)}
    run = {"t": a[:, idx["time_ms"]] / 1000.0}
    for short, name in COLS.items():
        if name in idx:
            run[short] = a[:, idx[name]]
    run["tau_rotor"] = K_T * ffill(run["i_q"])
    # The target in effect: torque under the torque method, else the velocity
    # one; what the active window and the signed means follow.
    target = np.nan_to_num(ffill(run["tau_set"]))
    if not target.any() and "omega_set" in run:
        target = np.nan_to_num(ffill(run["omega_set"]))
    run["target"] = target
    return run


REST_S = 0.5    # a target at zero this long separates two runs in one capture
TRAIL_S = 0.6   # kept after a run's last target: the coast-down


def split_runs(whole):
    # One capture may hold several runs: each span of nonzero target, with its
    # lead and its trail, becomes a run with time rebased to its own start.
    t, target = whole["t"], whole["target"]
    on = np.flatnonzero(target != 0)
    if not len(on):
        return []
    dt = np.median(np.diff(t))
    breaks = np.flatnonzero(np.diff(t[on]) > REST_S)
    starts = np.concatenate([[on[0]], on[breaks + 1]])
    ends = np.concatenate([on[breaks], [on[-1]]])
    runs = []
    for i, (s, e) in enumerate(zip(starts, ends)):
        lo = max(0, s - int(LEAD_S / dt))
        hi = min(len(t), e + int(TRAIL_S / dt), starts[i + 1] - int(LEAD_S / dt) if i + 1 < len(starts) else len(t))
        run = {k: v[lo:hi] for k, v in whole.items()}
        run["t"] = run["t"] - t[s] + LEAD_S
        runs.append(run)
    return runs


runs = {}
for source in ("sil", "bench"):
    for f in sorted((HERE / source).glob("*.csv")):
        m = RUN_NAME.match(f.name)
        if not m:
            print("skipping", f.name)
            continue
        scenario, design, j, tc, jcfg, rep = m.groups()
        parts = split_runs(load(f))
        for i, run in enumerate(parts):
            tag = (rep or "1") + (f".{i + 1}" if len(parts) > 1 else "")
            key = (source, scenario, design, float(j) if j else None, float(tc) if tc else None,
                   float(jcfg) if jcfg else None, tag if rep or len(parts) > 1 else None)
            run["design"] = design
            runs[key] = run
for p in sorted(HERE.glob("sil/provenance_*.txt")):
    print(p.name + ":", " | ".join(p.read_text().split("\n")[:3]))
print(f"{len(runs)} runs")


def label(key):
    source, scenario, design, j, tc, jcfg, rep = key
    return (design + (f" J×{j:g}" if j else "") + (f" Tc×{tc:g}" if tc else "")
            + (f" Jcfg{jcfg:g}" if jcfg else "") + (f" r{rep}" if rep else ""))


def configured_j(key):
    # The estimator's configured inertia for a run: the board's, unless the
    # capture's tag names the value it was flashed with.
    return key[5] * 1e-4 if key[5] else J_CFG

## Metrics

Over the active window — the first nonzero target to its last — per run.
The per-cycle peaks sit on the encoder noise (about ±0.1°, RMS 0.04°), so
the designs separate in the signed means: each error sample weighted by the
sign of the target in effect and averaged over the cycles under real torque,
where the delivered K_t·I_q (10 ms mean) is past half the run's peak target.
Under acceleration A lags behind the plant, so its signed bias is negative;
B with a wrong J errs either way.

- **SIL only.** The per-cycle angle error θ̂ − θ_plant: its signed mean under
  torque, peak and RMS, in degrees; the velocity error ω̂ − ω_plant, signed
  mean under torque and peak; and the velocity lag, the shift of ω̂ over
  the plant with the least mean square difference, searched both ways
  (positive: the estimate trails).
- **Both.** The correction residual e = θ_m − θ̂ at each 1 kHz correction:
  signed mean under torque, peak, RMS, and its mean over the last 200 ms of
  the window; and ω̂ against the encoder's own Δθ/Δt, stamped at each
  interval's midpoint: mean error under torque and lag of the two
  10 ms-averaged series.
- **C.** The torque state T̂_d: its mean over the last 200 ms of the
  window, the load it settled on, in mNm; and, SIL only, its signed mean
  error under torque against the plant's unmodelled torque,
  J_cfg·α_plant − (K_t·I_q − b·ω_plant − T_c·sign ω_plant), what the
  firmware's model leaves for the state to carry.

In [ ]:
def wrap(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def active_window(run):
    on = np.flatnonzero(run["target"] != 0)
    return run["t"][on[0]], run["t"][on[-1]]


def corrections(run):
    # The 1 kHz correction instants: where the measured angle changes.
    return np.flatnonzero(np.diff(ffill(run["theta_m"])) != 0) + 1


def encoder_velocity(run):
    # The encoder's own velocity at each interval's midpoint, rad/s.
    k = corrections(run)
    theta, t = ffill(run["theta_m"])[k], run["t"][k]
    return 0.5 * (t[1:] + t[:-1]), wrap(np.diff(theta)) / np.diff(t)


def moving_mean(x, n):
    return np.convolve(x, np.ones(n) / n, mode="same")


def lag_ms(t, a, b, max_ms=20.0):
    # How far a trails b (negative: leads): the shift with the least mean
    # square difference, on a's grid.
    dt = np.median(np.diff(t))
    n = int(round(max_ms / 1000 / dt))
    best, best_mse = 0, np.inf
    for s in range(-n, n + 1):
        d = a[max(s, 0) : len(a) + min(s, 0)] - b[max(-s, 0) : len(b) + min(-s, 0)]
        mse = np.mean(d**2)
        if mse < best_mse:
            best, best_mse = s, mse
    return best * dt * 1000


def under_torque(run):
    # The cycles under real torque: the delivered K_t I_q, 10 ms-averaged to
    # drop the ripple, past half the run's peak target (torque runs) or past
    # 0.02 Nm (velocity runs). Collapses at the voltage ceiling.
    tau = moving_mean(np.nan_to_num(run["tau_rotor"]), 200)
    peak = np.abs(np.nan_to_num(ffill(run["tau_set"]))).max()
    return np.abs(tau) >= (0.5 * peak if peak > 0 else 0.02)


def signed_mean(err, run, mask):
    return np.mean(np.sign(run["target"][mask]) * err[mask])


def unmodelled_torque(run):
    # SIL only: the torque the firmware's model leaves for T_d, from the
    # plant's own acceleration (1 ms-smoothed velocity differentiated), Nm.
    alpha = np.gradient(moving_mean(run["omega_plant"], 20), run["t"])
    alpha[:20] = alpha[-20:] = np.nan   # the moving mean's zero-padded ends
    modelled = (np.nan_to_num(run["tau_rotor"]) - (B_VISC * run["omega_plant"])
                - (T_C * np.sign(run["omega_plant"])))
    return (J_CFG * alpha) - modelled


def metrics(run):
    w = active_window(run)
    m = (run["t"] >= w[0]) & (run["t"] <= w[1])
    on = under_torque(run) & m
    out = {}
    if "theta_plant" in run:
        err = np.degrees(wrap(run["theta_hat"] - run["theta_plant"]))
        verr = run["omega_hat"] - run["omega_plant"]
        out["angle signed °"] = signed_mean(err, run, on)
        out["angle peak °"] = np.abs(err[m]).max()
        out["angle rms °"] = np.sqrt(np.mean(err[m] ** 2))
        out["vel signed"] = signed_mean(verr, run, on)
        out["vel peak"] = np.abs(verr[m]).max()
        out["vel lag ms"] = lag_ms(run["t"][m], run["omega_hat"][m], run["omega_plant"][m])
    k = corrections(run)
    k = k[m[k]]
    e = np.degrees(ffill(run["e"]))
    out["e signed °"] = signed_mean(e, run, k[on[k]]) if on[k].any() else np.nan
    out["e peak °"] = np.abs(e[k]).max()
    out["e rms °"] = np.sqrt(np.mean(e[k] ** 2))
    out["e tail °"] = e[k][run["t"][k] >= w[1] - 0.2].mean()
    te, we = encoder_velocity(run)
    wh = np.interp(te, run["t"], ffill(run["omega_hat"]))
    sel = (te >= w[0]) & (te <= w[1])
    on_e = np.interp(te, run["t"], on.astype(float)) > 0.5
    sgn = np.sign(np.interp(te, run["t"], run["target"]))
    out["ω̂−enc signed"] = np.mean((sgn * (wh - we))[sel & on_e])
    out["ω̂−enc lag ms"] = lag_ms(te[sel], moving_mean(wh[sel], 10), moving_mean(we[sel], 10))
    if run["design"] == "C":
        td = ffill(run["tau_hat"])
        out["Td tail mNm"] = 1e3 * td[m][run["t"][m] >= w[1] - 0.2].mean()
        if "omega_plant" in run:
            out["Td err mNm"] = 1e3 * signed_mean(td - unmodelled_torque(run), run, on)
    if not np.nan_to_num(ffill(run["tau_set"])).any():
        # A velocity run: the loop's overshoot past its step, from the encoder.
        step = np.abs(run["target"]).max()
        out["overshoot %"] = 100 * (np.abs(moving_mean(we, 10)[sel]).max() - step) / step
    return out


table = {key: metrics(run) for key, run in runs.items()}
names = list(dict.fromkeys(n for row in table.values() for n in row))
print(f"{'run':32s}" + "".join(f"{n:>14s}" for n in names))
for key, row in table.items():
    print(f"{key[0] + ' ' + key[1] + ' ' + label(key):32s}" + "".join(f"{row.get(n, float('nan')):14.3f}" for n in names))

## Residual decomposition

On the bench there is no plant truth, and the residual carries more than the
input term: the AS5048 reports its angle about 100 µs late (t_prop in the
datasheet), which reads as −ω·t_prop with the same sign as a J set too low,
and any unmodelled load (Coulomb friction, a hand on the shaft) leaves a
constant or sign-of-velocity term. The SIL encoder model has no t_prop, so
SIL shows none of that.

A least-squares fit at the correction instants over the active window
separates them: e ≈ c_a·a_m + c_ω·ω̂ + c_c·sign(ω̂) + c_0, with a_m the
model acceleration from the measured current, (K_t·I_q − b·ω̂)/J_cfg, which
both sides have and which B feeds its advance. The plant accelerates at
a_m / f for f = J_plant / J_cfg, so the residual the loop cancels is
a_m·(1/f − 1)/K_i under B and a_m/(f·K_i) under A: with g = K_i·c_a,
**f = 1/(1 + g)** for B and **f = 1/g** for A, the plant's inertia over the
configured one either way (K_t taken as right; Coulomb friction folds in as
a few percent). **t_prop** = −c_ω is the encoder's delay, and c_c, c_0 are
the load terms, in degrees. The reversal scenario is what makes a_m and ω̂
separable.

Under C the steady part of the residual is what T̂_d absorbs, so the same
fit runs on the torque state instead: T̂_d ≈ s·J_cfg·a_m + c_c·sign(ω̂) + c_0.
The plant accelerates at a_m/f while the model says a_m, and T̂_d makes up
J_cfg·a_m·(1/f − 1), so **f = 1/(1 + s)**; c_c and c_0 are the load terms
T̂_d holds, in mNm (in SIL, c_c should read the configured Coulomb friction
less the plant's, +5.6 mNm at the 1× variant; they separate only where the
run reverses). T̂_d follows the unmodelled torque with about a 2/p_d lag, so
the a_m regressor is first-order filtered at that time constant before the
fit: unfiltered, f reads 0.66 and 1.67 at the 0.5× and 2× variants;
filtered, 0.52 and 2.04. The residual e under C carries only the
transients, so its own fit is not run.

In [ ]:
def lagged(x, t, tau):
    # First-order response of x with time constant tau, on t's grid.
    y = np.zeros_like(x)
    a = np.median(np.diff(t)) / (tau + np.median(np.diff(t)))
    for k in range(1, len(x)):
        y[k] = y[k - 1] + a * (x[k] - y[k - 1])
    return y


def decompose(run, design, j_cfg):
    w = active_window(run)
    k = corrections(run)
    k = k[(run["t"][k] >= w[0]) & (run["t"][k] <= w[1])]
    t = run["t"][k]
    e = ffill(run["e"])[k]
    omega = ffill(run["omega_hat"])[k]
    a_model = (moving_mean(np.nan_to_num(run["tau_rotor"]), 20)[k] - B_VISC * omega) / j_cfg
    if design == "C":
        td = ffill(run["tau_hat"])[k]
        X = np.column_stack([lagged(j_cfg * a_model, t, 2 / P_D), np.sign(omega), np.ones_like(t)])
        c, *_ = np.linalg.lstsq(X, td, rcond=None)
        f = 1 / (1 + c[0])
        return {
            "f = J_plant/J_cfg": f,
            "J_plant e-4": f * j_cfg * 1e4,
            "Td_c mNm": 1e3 * c[1],
            "Td_0 mNm": 1e3 * c[2],
            "Td fit rms mNm": 1e3 * np.sqrt(np.mean((td - X @ c) ** 2)),
        }
    X = np.column_stack([a_model, omega, np.sign(omega), np.ones_like(t)])
    c, *_ = np.linalg.lstsq(X, e, rcond=None)
    resid = e - X @ c
    g = K_I * c[0]
    f = (1 / g) if design == "A" else 1 / (1 + g)
    return {
        "f = J_plant/J_cfg": f,
        "J_plant e-4": f * j_cfg * 1e4,
        "t_prop µs": -c[1] * 1e6,
        "c_c °": np.degrees(c[2]),
        "c_0 °": np.degrees(c[3]),
        "fit rms °": np.degrees(np.sqrt(np.mean(resid**2))),
    }


fit = {key: decompose(run, key[2], configured_j(key)) for key, run in runs.items()}
names = list(dict.fromkeys(n for row in fit.values() for n in row))
print(f"{'run':32s}" + "".join(f"{n:>16s}" for n in names))
for key, row in fit.items():
    print(f"{key[0] + ' ' + key[1] + ' ' + label(key):32s}" + "".join(f"{row.get(n, float('nan')):16.3f}" for n in names))

## Summary

Per source, scenario, design and plant variant: the median over the runs
and, as ±, half the spread between the first and third quartiles, for the
measures that decide — the signed residual under torque, the velocity lag
against the encoder, and the fitted inertia ratio. A run that stuck or
stalled shows as an outlier in the tables above and barely moves these.

In [ ]:
SUMMARY = [
    ("e signed °", table), ("e peak °", table), ("ω̂−enc lag ms", table), ("overshoot %", table),
    ("f = J_plant/J_cfg", fit), ("J_plant e-4", fit), ("c_c °", fit), ("t_prop µs", fit),
    ("Td tail mNm", table), ("Td err mNm", table), ("Td_c mNm", fit),
]
groups = {}
for key in runs:
    groups.setdefault(key[:6], []).append(key)
print(f"{'group':30s}{'n':>3s}" + "".join(f"{name:>20s}" for name, _ in SUMMARY))
for group, keys in groups.items():
    name = f"{group[0]} {group[1]} " + label(group + (None,))
    cells = []
    for metric, src in SUMMARY:
        v = np.array([src[k].get(metric, np.nan) for k in keys])
        if np.isnan(v).all():
            cells.append("")
            continue
        q1, q3 = np.nanpercentile(v, [25, 75])
        cells.append(f"{np.nanmedian(v):9.3f} ±{(q3 - q1) / 2:7.3f}")
    print(f"{name:30s}{len(keys):3d}" + "".join(f"{c:>20s}" for c in cells))

## Scenarios

Per scenario and source, the designs overlaid: the target and the torque the
current loop delivered; the velocity estimate against the plant (SIL, dashed
in the run's own colour — the plant differs per variant) or the encoder's
Δθ/Δt (bench); the angle error against the plant (SIL) with
`fw~est_rotor_002`'s 0.15° drawn; the torque state T̂_d where a design has
one, against the plant's unmodelled torque (SIL, dashed); and the
correction residual.

In [ ]:
def plot_scenario(source, scenario):
    keys = [k for k in runs if k[0] == source and k[1] == scenario]
    if not keys:
        return
    # One colour and one legend entry per group (design and variant); the
    # repeats of a group draw in its colour.
    groups_here = list(dict.fromkeys(k[:6] for k in keys))
    colour = {g: f"C{i}" for i, g in enumerate(groups_here)}
    seen = set()

    def style(key):
        g = key[:6]
        first = g not in seen
        seen.add(g)
        return {"color": colour[g], "label": label(g + (None,)) if first else None}

    has_plant = "theta_plant" in runs[keys[0]]
    has_td = any(runs[k]["design"] == "C" for k in keys)
    n = (4 if has_plant else 3) + (1 if has_td else 0)
    fig, axes = plt.subplots(n, 1, figsize=(11, 2.7 * n), sharex=True)
    ax = iter(axes)

    a = next(ax)
    r0 = runs[keys[0]]
    torque_run = np.nan_to_num(ffill(r0["tau_set"])).any()
    target_ax = a if torque_run else a.twinx()
    target_ax.plot(r0["t"], r0["target"], "k", lw=1, label="target")
    for key in keys:
        r = runs[key]
        a.plot(r["t"], r["tau_rotor"], lw=0.6, **style(key))
    a.set_ylabel("K_t I_q [Nm]" if not torque_run else "target / K_t I_q [Nm]")
    if not torque_run:
        target_ax.set_ylabel("target [rad/s]")
    a.legend(fontsize=8, ncol=4)

    seen.clear()
    a = next(ax)
    for key in keys:
        r = runs[key]
        s = style(key)
        a.plot(r["t"], ffill(r["omega_hat"]), lw=0.8, **s)
        if has_plant:
            a.plot(r["t"], r["omega_plant"], "--", color=s["color"], lw=0.7)
    if not has_plant:
        te, we = encoder_velocity(r0)
        a.plot(te, moving_mean(we, 10), "k--", lw=0.6, label="encoder Δθ/Δt, 10 ms mean")
    a.set_ylabel("velocity [rad/s]")
    a.legend(fontsize=8, ncol=4)

    if has_plant:
        seen.clear()
        a = next(ax)
        for key in keys:
            r = runs[key]
            a.plot(r["t"], np.degrees(wrap(r["theta_hat"] - r["theta_plant"])), lw=0.6, **style(key))
        for y in (0.15, -0.15):
            a.axhline(y, color="0.6", ls=":", lw=0.8)
        a.set_ylabel("θ̂ − θ_plant [°]")
        a.legend(fontsize=8, ncol=4)

    if has_td:
        seen.clear()
        a = next(ax)
        for key in keys:
            r = runs[key]
            if r["design"] != "C":
                continue
            s = style(key)
            a.plot(r["t"], 1e3 * ffill(r["tau_hat"]), lw=0.6, **s)
            if has_plant:
                a.plot(r["t"], 1e3 * unmodelled_torque(r), "--", color=s["color"], lw=0.5)
        a.set_ylabel("T̂_d [mNm]")
        a.legend(fontsize=8, ncol=4)

    seen.clear()
    a = next(ax)
    for key in keys:
        r = runs[key]
        k = corrections(r)
        a.plot(r["t"][k], np.degrees(ffill(r["e"])[k]), lw=0.6, **style(key))
    a.set_ylabel("residual e [°]")
    a.set_xlabel("t [s]")
    a.legend(fontsize=8, ncol=4)
    fig.suptitle(f"{scenario} — {source}")
    fig.tight_layout()
    return fig


for scenario in sorted({k[1] for k in runs}):
    for source in ("sil", "bench"):
        plot_scenario(source, scenario)

## Friction curve from the torque state

Under C, T̂_d is the torque the firmware's model leaves out, so at a steady
speed the plant's actual drag is the model's friction less T̂_d:
D(ω) = b·ω + T_c·sign(ω) − T̂_d. The delivered torque K_t·I_q at the same
steady speed is the independent check (at steady speed it equals the drag).
Quasi-steady samples only (|α̂| under 20 rad/s², α̂ from the 5 ms-smoothed
velocity estimate), binned 2 rad/s wide and centred on the even speeds, over every bench run of design C;
bins with fewer than 500 samples are dropped. Where the run coasts under a
torque below breakaway the rotor stands still and T̂_d reads the whole
applied torque, so the −2..0 bin is the stiction case, not a drag.


In [ ]:
def friction_curve(source="bench", design="C"):
    om, td, tau = [], [], []
    for key, run in runs.items():
        if key[0] != source or key[2] != design:
            continue
        omega = ffill(run["omega_hat"])
        alpha = np.gradient(moving_mean(omega, 100), run["t"])
        steady = np.abs(alpha) < 20
        om.append(omega[steady])
        td.append(ffill(run["tau_hat"])[steady])
        tau.append(np.nan_to_num(run["tau_rotor"])[steady])
    if not om:
        return None
    om, td, tau = map(np.concatenate, (om, td, tau))
    rows = []
    for lo in range(-23, 24, 2):   # bins centred on the even speeds, the 10 rad/s hold in one
        m = (om >= lo) & (om < lo + 2) & ~np.isnan(td)
        if m.sum() < 500:
            continue
        w = om[m].mean()
        model = B_VISC * w + T_C * np.sign(w)
        rows.append((w, m.sum(), 1e3 * model, 1e3 * td[m].mean(), 1e3 * td[m].std(), 1e3 * (model - td[m].mean()), 1e3 * tau[m].mean()))
    return np.array(rows)


curve = friction_curve()
if curve is not None:
    print(f"{'ω̂ rad/s':>9s} {'n':>7s} {'model':>7s} {'T̂_d':>7s} {'T̂_d sd':>7s} {'drag':>7s} {'K_t I_q':>8s}   (mNm)")
    for w, n, model, td_m, td_s, drag, tau_m in curve:
        print(f"{w:9.1f} {int(n):7d} {model:+7.1f} {td_m:+7.1f} {td_s:7.1f} {drag:+7.1f} {tau_m:+8.1f}")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(curve[:, 0], curve[:, 2], "k:", lw=1, label="configured model b·ω + T_c·sign ω")
    ax.plot(curve[:, 0], curve[:, 5], "C3o-", lw=1, ms=4, label="drag = model − T̂_d")
    ax.plot(curve[:, 0], curve[:, 6], "C0x", ms=5, label="K_t I_q (steady torque)")
    ax.axhline(0, color="0.7", lw=0.6)
    ax.set_xlabel("ω̂ [rad/s]")
    ax.set_ylabel("torque [mNm]")
    ax.legend(fontsize=8)
    fig.suptitle("friction curve — bench, design C")
    fig.tight_layout()


## Decision

The criteria and the findings live in `README.md` (*Decision*, *Findings*);
this notebook is their evidence.